# Lesson 4.5 — Optimizing Token, Latency, and Cost

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Apply levers that cut tokens, latency, and cost without wrecking quality
- Model the savings of each optimization
- Prioritize optimizations by impact vs risk

**You will produce:** A `cost_model` comparing baseline vs optimized configurations.

## Concepts

Optimization is a *guarded* activity: every lever that saves money can cost quality, so you re-run the eval harness after each change. Core levers:

- **Right-size the model** (Haiku for simple steps) — biggest cost lever.
- **Prompt caching** — reuse stable prefixes for cost + latency (Lesson 2.4).
- **Trim context** — retrieve fewer/better chunks; drop dead few-shot examples.
- **Cap output tokens** and avoid unnecessary chain-of-thought in the final payload.
- **Route** — cheap model first, escalate only hard cases (cascade).

The architect quantifies each lever's savings and ranks by **impact vs risk**, then validates quality is still above the gate from Lesson 4.1. This operationalizes the Domain 4 objective of optimizing cost-performance trade-offs.

> **Reference:** [Anthropic — Reducing latency & managing cost](https://docs.anthropic.com/en/docs/build-with-claude/reducing-latency)

In [ ]:
# !pip install anthropic
import os, json, time, random

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        # Deterministic mock: echo a label when the prompt asks for classification
        return f"[MOCK {model}] {prompt[:60]}"
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Build a cost model for a baseline config

We compute monthly cost from volume and tokens. This is the number stakeholders actually care about and the baseline every optimization is measured against.

In [ ]:
PRICE = {"haiku":{"in":0.0008,"out":0.004},
         "sonnet":{"in":0.003,"out":0.015},
         "opus":{"in":0.015,"out":0.075}}  # placeholders; verify at https://docs.anthropic.com

def monthly_cost(model, in_tok, out_tok, reqs_per_day, days=30, cached_frac=0.0, cache_discount=0.9):
    p = PRICE[model]
    eff_in = in_tok*(1-cached_frac) + in_tok*cached_frac*(1-cache_discount)
    per_req = eff_in/1000*p['in'] + out_tok/1000*p['out']
    return per_req * reqs_per_day * days

base = monthly_cost('sonnet', in_tok=6000, out_tok=400, reqs_per_day=20000)
print(f"baseline monthly cost: ${base:,.0f}")

## Code-Along: Step 2 — Compare optimizations and rank by savings

We model caching the 6k-token static prefix, trimming context, and routing simple cases to Haiku, then rank the levers by dollar savings.

In [ ]:
opts = {
  "prompt caching (90% prefix cached)": monthly_cost('sonnet',6000,400,20000,cached_frac=0.9),
  "trim context to 3000 in-tokens":     monthly_cost('sonnet',3000,400,20000),
  "route 70% simple cases to haiku":
       0.7*monthly_cost('haiku',6000,400,20000) + 0.3*monthly_cost('sonnet',6000,400,20000),
}
print(f"baseline: ${base:,.0f}\n")
for name, c in sorted(opts.items(), key=lambda kv: kv[1]):
    print(f"{name:38} ${c:,.0f}  (saves ${base-c:,.0f}, {100*(base-c)/base:.0f}%)")
print("\nAlways re-run the eval gate (Lesson 4.1) after each optimization.")

## Exercise

Combine levers: model caching + context trim + routing *together* and estimate the stacked savings (careful not to double-count). Then add a **quality risk** rating (low/med/high) to each lever and produce an impact-vs-risk ranking — recommend which to ship first and which needs an A/B test (Lesson 4.3) before rollout.

In [ ]:
# TODO: estimate combined savings of caching + trim + routing
# TODO: tag each lever with a quality-risk level and rank by impact/risk
# TODO: recommend ship order and which levers require an A/B test first

## Key Takeaways

- Biggest cost lever is right-sizing the model; caching cuts both cost and latency.
- Trim context and cap output tokens — fewer tokens, less cost and latency.
- Cascade routing sends only hard cases to expensive models.
- Re-run the eval gate after every optimization; rank levers by impact vs quality risk.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.